# SANAD Adaptive v0.2 — ESPARGOS Real CSI

**Research only.** Compares v0.1 covariance versus v0.2 CSI spectral-phase fusion. Automatically saves all results to Google Drive. Frozen RC3 is not used or modified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display, Image
import subprocess, sys, zipfile, datetime, os, json, hashlib
BASE = Path('/content/drive/MyDrive/SANAD_Adaptive')
ARCHIVE = BASE / 'Prototype_v02' / 'SANAD_Adaptive_v02.zip'
assert ARCHIVE.is_file(), f'Archive not found: {ARCHIVE}'
WORK = Path('/content/SANAD_Adaptive_v02')
WORK.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    assert not any(n.startswith('/') or '..' in Path(n).parts for n in archive.namelist())
    archive.extractall(WORK)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(WORK/'requirements-espargos.txt')])
subprocess.check_call([sys.executable,'-m','pip','install','-q','pytest>=8,<10'])
print('Loaded:',ARCHIVE)


In [ ]:
env=dict(os.environ,PYTHONPATH=str(WORK))
subprocess.check_call([sys.executable,'-m','pytest','-q',str(WORK/'tests')],env=env)


In [ ]:
# Download the two small original TFRecord files. The previous v0.1 dataset
# SHA-256 hashes will be recorded and can be compared for source consistency.
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DEST=BASE/'Results'/'ESPARGOS_V02'/STAMP
DEST.mkdir(parents=True,exist_ok=False)
DATA=Path('/content/SANAD_Adaptive_Datasets');DATA.mkdir(exist_ok=True)
cmd=[sys.executable,str(WORK/'experiments'/'run_espargos_v02.py'),
     '--download','--data-dir',str(DATA),'--output',str(DEST),'--window','16']
print('Running A/B research test; saving directly to:',DEST)
subprocess.check_call(cmd,env=env)


In [ ]:
# Verify source hashes and write integrity manifest after full completion.
summary=json.loads((DEST/'adaptive_v02_report.json').read_text())
print(json.dumps({key:summary[key] for key in ('status','static','walking','injections','limitations')},indent=2))
if (DEST/'adaptive_v02_comparison.png').is_file():display(Image(filename=str(DEST/'adaptive_v02_comparison.png')))
EXPECTED={
 'static':'cf2145a6fbca9fb1ba1c785138d07c8d803ba26205f4e18cea7e99e9928d8d73',
 'walking':'65b54e01d1a74b60b911c1f2686f61676204d8449741913b7504b1ee84b3206e'}
actual=summary['sources_sha256']
assert actual==EXPECTED, f'Source file hash mismatch; investigate before comparing runs: {actual}'
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
manifest={f.name:{'sha256':file_sha(f),'size_bytes':f.stat().st_size}
          for f in DEST.iterdir() if f.is_file()}
(DEST/'RESULTS_SHA256.json').write_text(json.dumps(manifest,indent=2))
print('OUTPUT SAVED AND HASHED IN GOOGLE DRIVE:',DEST)
